# Stage 1: Frozen SmolVLA on LIBERO
Reuse an existing single-GPU runtime. Run cells in order; stop on errors. No training. Smoke is excluded from baseline statistics.

## 1. Verify the existing GPU and prepare the workspace

In [ ]:
import sys, subprocess, pathlib
import torch
print(sys.version, torch.__version__, torch.version.cuda)
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0), torch.cuda.get_arch_list())
x = torch.ones(64, 64, device="cuda")
y = x @ x
torch.cuda.synchronize()
assert torch.all(y == 64).item()
print("Real CUDA matrix multiplication passed")
repo = pathlib.Path("/content/cs6758-vla-project")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/wcysunrise/cs6758-vla-project.git", str(repo)], check=True)
py = repo / ".venv/bin/python"
from datetime import datetime, timezone
run_dir = repo / "results/runs" / datetime.now(timezone.utc).strftime("reproduction_%Y%m%dT%H%M%S_%fZ")
run_dir.mkdir(parents=True, exist_ok=False)
print("Fresh run directory:", run_dir)
def run(args):
    subprocess.run([str(py), *args], cwd=repo, check=True)


## 2. Install isolated dependencies
The system PyTorch remains intact. LeRobot 0.5.1 requires torch <2.11, so this venv uses the compatible pinned CUDA 12.6 stack.

In [ ]:
if not py.exists():
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(repo / ".venv")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "--python", str(py), "install", "pip"], check=True)
run(["-m", "pip", "install", "torch==2.7.1", "torchvision==0.22.1", "--index-url", "https://download.pytorch.org/whl/cu126"])
run(["-m", "pip", "install", "-r", "requirements.txt"])
run(["-m", "pip", "check"])
run(["scripts/check_environment.py", "--output-dir", str(run_dir)])
with (run_dir / "installed_versions.txt").open("w") as f:
    subprocess.run([str(py), "-m", "pip", "freeze"], stdout=f, check=True)


## 3. Download the exact pretrained checkpoint and inference assets
No demonstration/training dataset is downloaded. Revisions are pinned.

In [ ]:
run(["scripts/prepare_policy.py"])

## 4. Model loading and simulator initialization
The reusable evaluator strictly loads the frozen model and initializes the selected task in the smoke cell. The manifest records both; no separate random policy is constructed.

In [ ]:
run(["scripts/run_baseline.py", "--check-config"])

## 5. Execute one complete GPU smoke episode
Task failure is an acceptable smoke outcome. Infrastructure errors stop execution.

In [ ]:
run(["-u", "scripts/run_baseline.py", "--smoke", "--output-dir", str(run_dir)])


## 6. Evaluate five fixed initial states
Requires the complete smoke record and verified smoke video. Each notebook execution uses a fresh output directory; delivered results are preserved.


In [ ]:
run(["-u", "scripts/run_baseline.py", "--output-dir", str(run_dir),
     "--smoke-record", str(run_dir / "smoke/smoke.json")])
print((run_dir / "summary.json").read_text())


## 7. Export code and actual results before stopping the runtime

In [ ]:
import zipfile, hashlib
archive = pathlib.Path("/content") / (run_dir.name + "_artifacts.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ["scripts", "configs", "notebooks", "tests"]:
        for f in (repo / folder).rglob("*"):
            if f.is_file() and "__pycache__" not in f.parts:
                z.write(f, f.relative_to(repo))
    for f in run_dir.rglob("*"):
        if f.is_file():
            z.write(f, pathlib.Path("results") / f.relative_to(run_dir))
    for name in ["README.md", "SETUP.md", "requirements.txt", ".gitignore"]:
        z.write(repo / name, name)
print("ZIP SHA256:", hashlib.sha256(archive.read_bytes()).hexdigest())
from google.colab import files
files.download(str(archive))


## 8. Inspect the genuine episode video
Stop the GPU runtime in Runtime management after exporting artifacts.

In [ ]:
from IPython.display import Video, display
display(Video(str(run_dir / "episode_001.mp4"), embed=True))
